In [48]:
import pandas as pd 
import numpy as np 
import time
from tqdm import tqdm  # ✅ add tqdm
import ollama
import re
import json
import concurrent.futures
import psutil
import threading

In [3]:

# def extract_json(text):
#     # remove code fences if they exist
#     text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())  # remove leading ```json or ```
#     text = re.sub(r"```$", "", text.strip())              # remove trailing ```
    
#     # extract first JSON object if multiple
#     match = re.search(r"\{.*\}", text, re.DOTALL)
#     if match:
#         return match.group(0)
#     return text


In [4]:
# dataframe_name = 'DividedDataset/resume_till_100_rows.csv'

In [5]:
# temp_df = pd.read_csv(dataframe_name)

In [11]:
dataframe_name = 'DividedDataset/resume_till_400_rows.csv'
temp_df = pd.read_csv(dataframe_name)



def extract_json(text):
    # remove code fences if they exist
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())  # remove leading ```json or ```
    text = re.sub(r"```$", "", text.strip())              # remove trailing ```
    
    # extract first JSON object if multiple
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text




atscoretemp = []   # make sure list is initialized
start_time = time.time()

for i in tqdm(range(len(temp_df)), desc="Processing resumes", unit="resume"):
    resume = temp_df.iloc[i,0]
    job    = temp_df.iloc[i,1]

    prompt_content = f"""
    You are an expert resume analyst. Your task is to analyze a candidate's resume and a job description to determine their fit. You MUST ONLY provide a JSON object in your response. Do not include any other text or conversational phrases.

    [JOB DESCRIPTION]
    {job}

    [CANDIDATE RESUME]
    {resume}

    Analyze the candidate's skills and experience against the job description.
    Provide a final match score on a scale from 1 to 100, where 1 is no match and 100 is a perfect match.
    
    Format your output strictly as a JSON object with the following keys:
    {{
      "match_score": int
    }}
    """

    response = ollama.chat(
        model='gemma3',
        messages=[{'role': 'user', 'content': prompt_content}]
    )

    try:
        raw_content = response['message']['content']
        json_str = extract_json(raw_content)   # 🧹 clean response
        ats_score_data = json.loads(json_str)
    
        match_score = ats_score_data.get('match_score', 'N/A')
        atscoretemp.append(match_score)

    except json.JSONDecodeError as e:
        print(f"\nError parsing JSON response at index {i}: {e}")
        print("Raw response:", response['message']['content'])


temp_df['new_ats'] = atscoretemp
temp_df.to_csv(dataframe_name, index=False, encoding='utf-8')

end_time = time.time()
elapsed_time = end_time - start_time
print(f"\n✅ Finished in {elapsed_time:.2f} seconds")


Processing resumes: 100%|██████████████████████████████████████████████████████| 101/101 [1:01:03<00:00, 36.27s/resume]


✅ Finished in 3663.54 seconds


In [32]:
neww_df = pd.read_csv("DividedDataset/resume_till_1560_rows.csv")


In [33]:
neww_df.head()

,resume,jd,ats_score,new_ats
0,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nPosition: Junior PHP Dev...,51.84,78
1,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Summary:\n\nWe are s...,23.41,65
2,Vishal Kumar \nFull Stack Developer \nMy goal ...,Job Title: Sr. Website UI/UX Designer and Deve...,49.52,68
3,Vishal Kumar \nFull Stack Developer \nMy goal ...,We are looking for a passionate and enthusiast...,41.10,85
4,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Title: Software Engi...,32.23,85


In [42]:
# final_df = pd.read_csv("DividedDataset/resume_till_100_rows.csv")
# for i in range(200,1600, 100):
#     dataset_path = f"DividedDataset/resume_till_{i}_rows.csv"
#     temp = pd.read_csv(dataset_path)
#     final_df = pd.concat([final_df, temp])

In [35]:
dfs = []

# Loop through all files
for i in range(100, 1600, 100):
    dataset_path = f"DividedDataset/resume_till_{i}_rows.csv"
    dfs.append(pd.read_csv(dataset_path))

# Concatenate all at once
final_df = pd.concat(dfs, ignore_index=True)

In [38]:
path = "DividedDataset/resume_till_1560_rows.csv"

In [39]:
temp = pd.read_csv(path)

In [40]:
final_df = pd.concat([final_df, temp])

In [41]:
final_df.shape

(1575, 4)

In [44]:
final_df.isnull().sum()

resume       0
jd           0
ats_score    0
new_ats      0
dtype: int64

In [45]:
final_df.to_csv("final_dataframe.csv", index = False, encoding = 'utf-8')

In [56]:
temp_df = pd.read_csv("final_dataframe.csv")

In [58]:
temp_df.head()

,resume,jd,new_ats
0,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nPosition: Junior PHP Dev...,75
1,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Summary:\n\nWe are s...,65
2,Abhishek Chaudhary \n ac5712916@gmail.com |...,Job Title: Sr. Website UI/UX Designer and Deve...,65
3,Abhishek Chaudhary \n ac5712916@gmail.com |...,We are looking for a passionate and enthusiast...,85
4,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Title: Software Engi...,65


In [55]:
temp_df.to_csv("final_dataframe.csv", index = False, encoding = 'utf-8')